In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
from torch.utils.data import DataLoader
import onnx
import cv2

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from model import Net
from utils.config import Config
from datasets import TrainDataset

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
%pwd

'/home/jetson/Projects/DL-Based-UAV-Positioning-in-Blockage-Aware-Channel-Model/src/edge_device'

In [3]:
cfg = Config(num_users=6)

csv_path = os.path.join('..', 'train_model', 'result', 'data', 'gn_coords_6.csv')
df = pd.read_csv(csv_path, header=None)
df = df.drop(columns=df.columns[2::3])
x = torch.tensor(df.values, dtype=torch.float32, device=cfg.device)
x_scaled = cfg.scaler.transform(x.cpu())
x_dataset = TrainDataset(x_scaled, dtype=torch.float32).to(cfg.device)
x_dataloader = DataLoader(x_dataset, batch_size=1, shuffle=False)

model = Net(x_dataset.x.shape[1], 1024, 4, output_N=2).to(cfg.device)
model.load_state_dict(torch.load(os.path.join('..', 'train_model', 'result', 'models', 'num_gu', 'best_num_gu_6.pt'), map_location=cfg.device))

model.eval()


Net(
  (layers): ModuleList(
    (0): Linear(in_features=12, out_features=1024, bias=True)
    (1-4): 4 x Linear(in_features=1024, out_features=1024, bias=True)
  )
  (dropouts): ModuleList(
    (0-3): 4 x Dropout(p=0.3, inplace=False)
  )
  (batches): ModuleList(
    (0-3): 4 x BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (prelus): ModuleList(
    (0-3): 4 x PReLU(num_parameters=1)
  )
  (output): Linear(in_features=1024, out_features=2, bias=True)
)

In [4]:
model_path = "models"

if not os.path.exists(model_path):
    os.makedirs(model_path)
    

torch.onnx.export(
    model, x, os.path.join(model_path, 'model.onnx'),
    input_names=["input"], output_names=["output"],
    opset_version=13, do_constant_folding=True,
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    verbose=True
    )

Exported graph: graph(%input : Float(*, 12, strides=[1, 10000], requires_grad=0, device=cuda:0),
      %layers.0.weight : Float(1024, 12, strides=[12, 1], requires_grad=1, device=cuda:0),
      %layers.0.bias : Float(1024, strides=[1], requires_grad=1, device=cuda:0),
      %layers.1.weight : Float(1024, 1024, strides=[1024, 1], requires_grad=1, device=cuda:0),
      %layers.1.bias : Float(1024, strides=[1], requires_grad=1, device=cuda:0),
      %layers.2.weight : Float(1024, 1024, strides=[1024, 1], requires_grad=1, device=cuda:0),
      %layers.2.bias : Float(1024, strides=[1], requires_grad=1, device=cuda:0),
      %layers.3.weight : Float(1024, 1024, strides=[1024, 1], requires_grad=1, device=cuda:0),
      %layers.3.bias : Float(1024, strides=[1], requires_grad=1, device=cuda:0),
      %batches.0.weight : Float(1024, strides=[1], requires_grad=1, device=cuda:0),
      %batches.0.bias : Float(1024, strides=[1], requires_grad=1, device=cuda:0),
      %batches.0.running_mean : Float

In [5]:
!pip install onnx onnx-tf

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
!onnx2tf -i models/model.onnx -o models/tf_model -b 1 -ois "input:1,12"


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                    ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ BatchNormalization │ 4              │ 4                │
│ Constant           │ 30             │ 30               │
│ Gemm               │ 5              │ 5                │
│ PRelu              │ 4              │ 4                │
│ Sigmoid            │ 1              │ 1                │
│ Model Size         │ 12.2MiB        │ 12.2MiB          │
└────────────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                    ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ BatchNormalization │ 4              │ 4          

In [7]:
from datasets import BlockageDataset
from obstacles import create_obstacle_data
from utils.config import set_random_seed

set_random_seed()
obstacle_ls, obst_tensor = create_obstacle_data()

set_random_seed()
temp_dataset = BlockageDataset(10000, obstacle_ls, cfg=cfg)
temp_data = temp_dataset.gnd_nodes[:, :, :-1].reshape(-1, cfg.num_users*2).cpu().numpy()
samples = cfg.scaler.transform(temp_data)
temp_data

def representative_data_gen():
    for x in samples:
        yield [x.astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_saved_model("models/tf_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
open("models/model_int8_full.tflite","wb").write(tflite_model)

Generating ground nodes: 100%|██████████| 10000/10000 [00:03<00:00, 2907.50it/s]
W0000 00:00:1770979474.698191    2922 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1770979474.698268    2922 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-02-13 19:44:34.700020: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/tf_model
2026-02-13 19:44:34.704337: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-02-13 19:44:34.704445: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/tf_model
2026-02-13 19:44:34.717226: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-02-13 19:44:34.718091: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-02-13 19:44:34.742235: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: mod

3279792

In [8]:
interp = tf.lite.Interpreter(model_path="models/model_int8_full.tflite")
interp.allocate_tensors()
print(interp.get_input_details(), interp.get_output_details())

[{'name': 'serving_default_input:0', 'index': 0, 'shape': array([ 1, 12], dtype=int32), 'shape_signature': array([ 1, 12], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.003921537660062313, -128), 'quantization_parameters': {'scales': array([0.00392154], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}] [{'name': 'PartitionedCall:0', 'index': 24, 'shape': array([1, 2], dtype=int32), 'shape_signature': array([1, 2], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.00390625, -128), 'quantization_parameters': {'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [9]:
interpreter = tf.lite.Interpreter(model_path="models/model_int8_full.tflite")  # 경로/파일명 수정
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

# 2) 입력 x_scaled(float32) → int8로 양자화
x_float = np.asarray(x_scaled, dtype=np.float32)
in_scale, in_zero = inp["quantization"]  # (scale, zero_point)
x_int8 = np.clip(np.round(x_float / in_scale + in_zero), -128, 127).astype(np.int8)

# 3) 추론
interpreter.set_tensor(inp["index"], x_int8[0].reshape(1,-1))
interpreter.invoke()
y_int8 = interpreter.get_tensor(out["index"])

# 4) 출력 역양자화(int8 → float32 원복)
out_scale, out_zero = out["quantization"]
y_float = (y_int8.astype(np.float32) - out_zero) * out_scale

y_float*200-100

array([[-28.90625, -75.     ]], dtype=float32)

In [10]:
!bash build_engine.sh

Using ONNX:   /home/jetson/Projects/DL-Based-UAV-Positioning-in-Blockage-Aware-Channel-Model/src/edge_device/models/model.onnx
Saving engine: /home/jetson/Projects/DL-Based-UAV-Positioning-in-Blockage-Aware-Channel-Model/src/edge_device/models/model_fp16.engine
trtexec bin : /usr/src/tensorrt/bin/trtexec
&&&& RUNNING TensorRT.trtexec [TensorRT v8602] # /usr/src/tensorrt/bin/trtexec --onnx=/home/jetson/Projects/DL-Based-UAV-Positioning-in-Blockage-Aware-Channel-Model/src/edge_device/models/model.onnx --saveEngine=/home/jetson/Projects/DL-Based-UAV-Positioning-in-Blockage-Aware-Channel-Model/src/edge_device/models/model_fp16.engine --fp16
[02/13/2026-19:44:55] [I] === Model Options ===
[02/13/2026-19:44:55] [I] Format: ONNX
[02/13/2026-19:44:55] [I] Model: /home/jetson/Projects/DL-Based-UAV-Positioning-in-Blockage-Aware-Channel-Model/src/edge_device/models/model.onnx
[02/13/2026-19:44:55] [I] Output:
[02/13/2026-19:44:55] [I] === Build Options ===
[02/13/2026-19:44:55] [I] Max batch: exp

In [11]:
import numpy as np
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # CUDA 컨텍스트 초기화

ENGINE_PATH = "models/model_fp16.engine"  # 엔진 경로

def load_engine(path):
    logger = trt.Logger(trt.Logger.ERROR)
    runtime = trt.Runtime(logger)
    with open(path, "rb") as f:
        return runtime.deserialize_cuda_engine(f.read())

def infer(engine, x):  # x: np.float32, shape (1,12)
    ctx = engine.create_execution_context()

    # 바인딩 인덱스 확인(단일 입력/출력 가정)
    inp_idx, out_idx = 0, 1
    ctx.set_binding_shape(inp_idx, x.shape)

    # 출력 shape 확인
    out_shape = tuple(ctx.get_binding_shape(out_idx))

    # 디바이스 메모리 할당
    d_in = cuda.mem_alloc(x.nbytes)
    y = np.empty(out_shape, dtype=np.float32)
    d_out = cuda.mem_alloc(y.nbytes)

    # H2D 복사 → 실행 → D2H 복사
    cuda.memcpy_htod(d_in, x)
    ctx.execute_v2([int(d_in), int(d_out)])
    cuda.memcpy_dtoh(y, d_out)

    # 메모리 해제(선택)
    d_in.free()
    d_out.free()
    return y

if __name__ == "__main__":
    engine = load_engine(ENGINE_PATH)
    x = x_float[0].reshape(1, -1).copy(order='C')
    y = infer(engine, x)
    print(y.shape, y.dtype, y*200-100)

(1, 2) float32 [[-28.844849 -75.115425]]


/tmp/ipykernel_2922/270283261.py:19: DeprecationWarning: Use set_input_shape instead.
  ctx.set_binding_shape(inp_idx, x.shape)
/tmp/ipykernel_2922/270283261.py:22: DeprecationWarning: Use get_tensor_shape instead.
  out_shape = tuple(ctx.get_binding_shape(out_idx))


In [12]:
type(x_float[0].reshape(1,-1)), type(np.random.rand(1, 12).astype(np.float32))

(numpy.ndarray, numpy.ndarray)